In [1]:
from audiocraft.utils import export
from audiocraft import train
xp = train.main.get_xp_from_sig('ea50bf50')
export.export_lm(xp.folder / 'checkpoint.th', '../outputs/my_audio_lm/state_dict.bin')
export.export_pretrained_compression_model('facebook/encodec_32khz', '../outputs/my_audio_lm/compression_state_dict.bin')

Dora directory: /tmp/audiocraft_shperry


In [2]:
import audiocraft.models
musicgen = audiocraft.models.MusicGen.get_pretrained('../outputs/my_audio_lm')


WRONG MODEL!!!!!!


In [3]:
musicgen.lm

LMModel(
  (cfg_dropout): ClassifierFreeGuidanceDropout(p=0.3)
  (att_dropout): AttributeDropout({})
  (condition_provider): ConditioningProvider(
    (conditioners): ModuleDict(
      (description): T5Conditioner(
        (output_proj): Linear(in_features=768, out_features=1024, bias=True)
      )
    )
  )
  (fuser): ConditionFuser()
  (emb): ModuleList(
    (0-3): 4 x ScaledEmbedding(2049, 1024)
  )
  (transformer): StreamingTransformer(
    (layers): ModuleList(
      (0-23): 24 x StreamingTransformerLayer(
        (self_attn): StreamingMultiheadAttention(
          (out_proj): Linear(in_features=1024, out_features=1024, bias=False)
        )
        (linear1): Linear(in_features=1024, out_features=4096, bias=False)
        (dropout): Dropout(p=0.0, inplace=False)
        (linear2): Linear(in_features=4096, out_features=1024, bias=False)
        (norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  

In [6]:
type(musicgen)

audiocraft.models.musicgen.MusicGen

In [ ]:
musicgen.compression_model

HFEncodecCompressionModel(
  (model): EncodecModel(
    (encoder): EncodecEncoder(
      (layers): ModuleList(
        (0): EncodecConv1d(
          (conv): ParametrizedConv1d(
            1, 64, kernel_size=(7,), stride=(1,)
            (parametrizations): ModuleDict(
              (weight): ParametrizationList(
                (0): _WeightNorm()
              )
            )
          )
        )
        (1): EncodecResnetBlock(
          (block): ModuleList(
            (0): ELU(alpha=1.0)
            (1): EncodecConv1d(
              (conv): ParametrizedConv1d(
                64, 32, kernel_size=(3,), stride=(1,)
                (parametrizations): ModuleDict(
                  (weight): ParametrizationList(
                    (0): _WeightNorm()
                  )
                )
              )
            )
            (2): ELU(alpha=1.0)
            (3): EncodecConv1d(
              (conv): ParametrizedConv1d(
                32, 64, kernel_size=(1,), stride=(1,)
          

In [16]:
from audiocraft.data.audio_dataset import AudioDataset

dataset = AudioDataset.from_meta("egs/birdset", channels=1)

In [17]:
dataset[0]

tensor([[-7.0900e-06,  1.2421e-05,  2.8561e-05,  ..., -4.2564e-04,
         -5.5145e-04, -6.2217e-04]])

In [ ]:
#Audio Tokens -> Cookbook
musicgen.compression_model.encode(dataset[0].unsqueeze(0).cuda())

(tensor([[[ 166,  778, 1436,  ...,  648,    8, 1454],
          [1534, 1922,   73,  ..., 1994, 2044, 1931],
          [2019, 2004, 1708,  ..., 1137, 1276,  970],
          [1903,  859, 2008,  ...,  339, 1185, 1670]]], device='cuda:0'),
 None)

In [21]:
cookbook = musicgen.compression_model.encode(dataset[0].unsqueeze(0).cuda())
musicgen.generate_audio(cookbook[0])

tensor([[[-0.0015, -0.0021, -0.0020,  ...,  0.0162,  0.0163,  0.0173]]],
       device='cuda:0')